In [76]:
!mkdir -p drive
!google-drive-ocamlfuse drive

/bin/bash: google-drive-ocamlfuse: command not found


In [0]:
import sys
sys.path.insert(0, 'drive/')

In [0]:
!pip install -q keras
!pip install -q tensorflow
!pip install -q numpy
!pip install -q pandas
!pip install -q nltk
!pip install -U -q PyDrive
!pip install -U -q sumeval


In [0]:
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# 1. Authenticate and create the PyDrive client.
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

# 2. Load a file by ID and create local file.
downloaded = drive.CreateFile({'id':'1KUQJtBbrlczw43i0eqzr4Fp5R9OICeRq'}) # replace fileid with Id of file you want to access
downloaded.GetContentFile('Reviews.csv') # now you can use export.csv 

downloaded2 = drive.CreateFile({'id':'1_M0ya_yrrNwTEEPXjXycDM73KM8CR1Ln'}) # replace fileid with Id of file you want to access
downloaded2.GetContentFile('glove.6B.100d.txt') # now you can use export.csv 

In [80]:
import tensorflow as tf
import numpy as np
import pandas as pd
import os as os
import re
import nltk
nltk.download('punkt')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [0]:
from tensorflow.python.keras.models import Model
from tensorflow.python.keras.layers import Input,Dense,GRU,Embedding,CuDNNGRU,CuDNNLSTM
from tensorflow.python.keras.optimizers import RMSprop
from tensorflow.python.keras.callbacks import ModelCheckpoint
from tensorflow.python.keras.preprocessing.text import Tokenizer
from tensorflow.python.keras.preprocessing.sequence import pad_sequences
from sumeval.metrics.rouge import RougeCalculator
from nltk.translate.bleu_score import sentence_bleu

In [0]:
reviews = pd.read_csv("Reviews.csv")
#news

In [83]:
reviews.shape

(568454, 10)

In [84]:
reviews.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [85]:
reviews.isnull().sum()

Id                         0
ProductId                  0
UserId                     0
ProfileName               16
HelpfulnessNumerator       0
HelpfulnessDenominator     0
Score                      0
Time                       0
Summary                   27
Text                       0
dtype: int64

In [0]:
reviews = reviews.dropna()
reviews = reviews.drop(['Id','ProductId','UserId','ProfileName','HelpfulnessNumerator','HelpfulnessDenominator','Score','Time'], 1)
reviews['Text']=reviews['Text'].fillna("")
reviews['Summary']=reviews['Summary'].fillna("")
reviews = reviews.reset_index(drop=True)

In [87]:
reviews.tail()

,Summary,Text
568406,Will not do without,Great for sesame chicken..this is a good if no...
568407,disappointed,I'm disappointed with the flavor. The chocolat...
568408,Perfect for our maltipoo,"These stars are small, so you can give 10-15 o..."
568409,Favorite Training and reward treat,These are the BEST treats for training and rew...
568410,Great Honey,"I am very satisfied ,product is as advertised,..."


In [88]:
reviews.Text[15]

"My daughter loves twizzlers and this shipment of six pounds really hit the spot. It's exactly what you would expect...six packages of strawberry twizzlers."

In [89]:
reviews.Summary[15]

'Lots of twizzlers, just what you expect.'

In [0]:
contractions = { 
"ain't": "am not",
"aren't": "are not",
"can't": "cannot",
"can't've": "cannot have",
"'cause": "because",
"could've": "could have",
"couldn't": "could not",
"couldn't've": "could not have",
"didn't": "did not",
"doesn't": "does not",
"don't": "do not",
"hadn't": "had not",
"hadn't've": "had not have",
"hasn't": "has not",
"haven't": "have not",
"he'd": "he would",
"he'd've": "he would have",
"he'll": "he will",
"he's": "he is",
"how'd": "how did",
"how'll": "how will",
"how's": "how is",
"i'd": "i would",
"i'll": "i will",
"i'm": "i am",
"i've": "i have",
"isn't": "is not",
"it'd": "it would",
"it'll": "it will",
"it's": "it is",
"let's": "let us",
"ma'am": "madam",
"mayn't": "may not",
"might've": "might have",
"mightn't": "might not",
"must've": "must have",
"mustn't": "must not",
"needn't": "need not",
"oughtn't": "ought not",
"shan't": "shall not",
"sha'n't": "shall not",
"she'd": "she would",
"she'll": "she will",
"she's": "she is",
"should've": "should have",
"shouldn't": "should not",
"that'd": "that would",
"that's": "that is",
"there'd": "there had",
"there's": "there is",
"they'd": "they would",
"they'll": "they will",
"they're": "they are",
"they've": "they have",
"wasn't": "was not",
"we'd": "we would",
"we'll": "we will",
"we're": "we are",
"we've": "we have",
"weren't": "were not",
"what'll": "what will",
"what're": "what are",
"what's": "what is",
"what've": "what have",
"where'd": "where did",
"where's": "where is",
"who'll": "who will",
"who's": "who is",
"won't": "will not",
"wouldn't": "would not",
"you'd": "you would",
"you'll": "you will",
"you're": "you are"
}

In [0]:
def clean_text(text,contradictions = True):
    # Convert words to lower case
    
    if type(text) is str:
        text = text.lower()
    
    # Replace contractions with their longer forms 
    if contradictions:
        text = text.split()
        new_text = []
        for word in text:
            if word in contractions:
                new_text.append(contractions[word])
            else:
                new_text.append(word)
        text = " ".join(new_text)
    
    # Format words and remove unwanted characters
    text = re.sub(r'\<a href', ' ', text)
    text = re.sub(r'https?:\/\/.*[\r\n]*', '', text, flags=re.MULTILINE)
    text = re.sub(r'&amp;', '', text) 
    #text = re.sub(r'[_"\-;%()|+&=*%.,!?:#$@\[\]/]', ' ', text)
    text = re.sub(r'<br\s*\/?>', '', text)
    text = re.sub(r'<br />', ' ', text)
    text = re.sub(r'[^-''.,;<>\\|+!?"-*_a-zA-Z0-9 \n\.]', ' ', text)
    
    #text = re.sub(r'\'', ' ', text)

    return text

In [92]:
clean_summaries = []

for summary in reviews.Summary:
    clean_summaries.append(clean_text(summary,contradictions = True))
print("Summaries are complete.")


Summaries are complete.


In [93]:
clean_texts = []
for text in reviews.Text:
    clean_texts.append(clean_text(text,contradictions = True))
print("Texts are complete.")



Texts are complete.


In [94]:
clean_texts[100]

'the mouth says, "how do i love thee, let me count the ways..."if you like apple products a must have item. the only draw back, shipping cost. these are very heavy.'

In [95]:
clean_summaries[100]

'taste wise it is a 6 star item'

In [0]:
mark_start = 'ssstarttoken '  #kelime haznesi içinde bulunmayan bir başlangıç tokeni veriyoruz.
#dekoder bu tokeni gördüğü zaman kelime üretmeye başlayacak.
mark_end = ' eeendtoken' #cümle bitiş tokeni dekoder cümlenin bitmesi karar verdiği zaman bu tokeni kullanacak
data_src = []
data_dest = []

data_src = clean_texts
for line in clean_summaries:
    data_dest.append(mark_start+line+mark_end)




In [97]:
print(data_src[100])
print(data_dest[100])


the mouth says, "how do i love thee, let me count the ways..."if you like apple products a must have item. the only draw back, shipping cost. these are very heavy.
ssstarttoken taste wise it is a 6 star item eeendtoken


In [0]:
#len(data_src)
class TokenizerWrap(Tokenizer): #tokenizer nesnesini alıyoruz.
    def __init__(self,texts,padding,reverse=False,num_words=None):
        Tokenizer.__init__(self,num_words=num_words)
        
        self.fit_on_texts(texts) #tokenizera textlerimizi veriyoruz.
        self.index_to_word = dict(zip(self.word_index.values(),self.word_index.keys())) #kelime haznesindeki kelime rakam değerlerini key value olarak
        #yer değiştirir.
        self.tokens = self.texts_to_sequences(texts) #cümleler tokenlara çevrilir.
        
        if reverse:
            self.tokens = [list(reversed(x)) for x in self.tokens] #encodera vereceğimiz tüm inputlar aynı uzunlukta olmalı
            truncating = 'pre' #truncating işlemi cümlenin başından veya sonundan pre ve post durumuna göre token atar.
        else:
            truncating = 'post' #cümlenin başı sonundan daha önemli olduğundan gelen texti ters çeviriyoruz.
            
        self.num_tokens = [len(x) for x in self.tokens]
        self.max_tokens = np.mean(self.num_tokens) + 2 * np.std(self.num_tokens) #RNN e vereceğimiz vectörün boyutlarını belirliyoruz.
        #bu ortalama ve standart sapmayla optimum bir büyüklük belirliyoruz.
        self.max_tokens = int(self.max_tokens)
        
        self.tokens_padded = pad_sequences(self.tokens,  #gelen input textte sıfırdan oluşan padding ekleyip boyutları eşit hale getiriyoruz.
                                          maxlen = self.max_tokens,
                                          padding=padding,
                                          truncating = truncating)
        
    def token_to_word(self,token):#sayının kelime karşılığını döndürür
        word = ' ' if token == 0 else self.index_to_word[token]
        return word
    
    def tokens_to_string(self,tokens): #verilen tokenlardan cümle döndürür
        words = [self.index_to_word[token] for token in tokens if token != 0]
        text = ' '.join(words)
        return text
    
    def text_to_tokens(self,text,padding,reverse = False): #verilen texti tokenlara dönüştürür
        tokens = self.texts_to_sequences([text])
        tokens = np.array(tokens)
        
        if reverse:
            tokens = np.flip(tokens,axis=1)
            truncating = 'pre'
        else:
            truncating = 'post'
            
        tokens = pad_sequences(tokens,
                              maxlen=self.max_tokens,
                              padding=padding,
                              truncating = truncating)
        return tokens

                

In [0]:
tokenizer_src = TokenizerWrap(texts = data_src, #encodera verilecek 
                             padding = 'pre',
                             reverse = True,
                             num_words=None)

In [0]:
tokenizer_dest = TokenizerWrap(texts = data_dest,#decodere verilecek
                             padding = 'post',
                             reverse = False,
                             num_words=None)

In [101]:
tokens_src = tokenizer_src.tokens_padded
tokens_dest = tokenizer_dest.tokens_padded
print(tokens_src.shape)
print(tokens_dest.shape)

(568411, 234)
(568411, 11)


In [102]:
tokens_dest[0]

array([ 1,  5, 68, 31, 29,  2,  0,  0,  0,  0,  0], dtype=int32)

In [103]:
tokenizer_dest.tokens_to_string(tokens_dest[0])

'ssstarttoken good quality dog food eeendtoken'

In [104]:
tokens_src[0]

array([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,   

In [105]:
tokenizer_src.tokens_to_string(tokens_src[0])

'most than better product this appreciates she and finicky is labrador my better smells it and meat processed a than stew a like more looks product the quality good of be to all them found have and products food dog canned vitality the of several bought have i'

In [106]:
token_start = tokenizer_dest.word_index[mark_start.strip()]
token_start

1

In [107]:
token_end = tokenizer_dest.word_index[mark_end.strip()]
token_end

2

In [0]:
encoder_input_data = tokens_src

In [0]:
decoder_input_data = tokens_dest[:,:-1] #sondan bir önceki tokena kadar verileri al
decoder_output_data = tokens_dest[:,1:] #1den sona kadar inputun 1 kaydırılmış hali

In [110]:
encoder_input_data[100]

array([    0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,

In [111]:
decoder_input_data[100]

array([   1,   24, 2991,   11,   15,    7,  469,  441,  224,    2],
      dtype=int32)

In [112]:
decoder_output_data[100]

array([  24, 2991,   11,   15,    7,  469,  441,  224,    2,    0],
      dtype=int32)

In [113]:
tokenizer_dest.tokens_to_string(decoder_input_data[100])

'ssstarttoken taste wise it is a 6 star item eeendtoken'

In [114]:
tokenizer_dest.tokens_to_string(decoder_output_data[100])

'taste wise it is a 6 star item eeendtoken'

In [0]:
num_encoder_words = len(tokenizer_src.word_index) + 1
num_decoder_words = len(tokenizer_dest.word_index) + 1

In [116]:
num_encoder_words

142812

In [117]:
num_decoder_words


35395

In [0]:
embedding_size = 100 #glove vektörlerinin uzunluğuyla aynı olmalı

In [0]:
word2vec = {} #eğitilmiş kelime glove kelime haznesini okuyoruz. Kelime anlamlarını tutuyor.
with open('glove.6B.100d.txt',encoding = 'UTF-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vec = np.array(values[1:], dtype='float32')
        word2vec[word] = vec


In [0]:
embedding_matrix = np.random.uniform(-1, 1, (num_encoder_words, embedding_size)) #önce boş bir matrix oluştur
for word,i in tokenizer_src.word_index.items():#kaynak textteki kelimeler glovedan gelen kelimeler içinde yoksa glovedan kelime alıyoruz.
    if i < num_encoder_words:
        embedding_vector = word2vec.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

In [121]:
embedding_matrix.shape


(142812, 100)

In [0]:
encoder_input = Input(shape=(None,),name='encoder_input')

In [0]:
encoder_embedding = Embedding(input_dim=num_encoder_words,
                             output_dim=embedding_size,
                             weights = [embedding_matrix],
                             trainable=True,
                             name='encoder_embedding')

In [0]:
state_size = 256

In [0]:
encoder_gru1 = CuDNNGRU(state_size,name="encoder_gru1",return_sequences=True)
encoder_gru2 = CuDNNGRU(state_size,name="encoder_gru2",return_sequences=True)
encoder_gru3 = CuDNNGRU(state_size,name="encoder_gru3",return_sequences=False)

In [0]:
def connect_encoder():
    layer = encoder_input
    layer = encoder_embedding(layer)
    layer = encoder_gru1(layer)
    layer = encoder_gru2(layer)
    layer = encoder_gru3(layer)
    
    encoder_output = layer
    
    return encoder_output

In [0]:
encoder_output = connect_encoder()

In [0]:
decoder_initial_state = Input(shape = (state_size,),name='decoder_initial_state')

In [0]:
decoder_input = Input(shape=(None,),name='decoder_input')

In [0]:
decoder_embedding = Embedding(input_dim = num_decoder_words,
                             output_dim = embedding_size,
                             name = 'decoder_embedding')

In [0]:
decoder_gru1 = CuDNNGRU(state_size, name='decoder_gru1',return_sequences=True)
decoder_gru2 = CuDNNGRU(state_size, name='decoder_gru2',return_sequences=True)
decoder_gru3 = CuDNNGRU(state_size, name='decoder_gru3',return_sequences=True)

In [0]:
decoder_dense = Dense(num_decoder_words,
                     activation='linear',
                     name='decoder_output')

In [0]:
def connect_decoder(initial_state):
    layer = decoder_input
    layer = decoder_embedding(layer)
    layer = decoder_gru1(layer,initial_state = initial_state)
    layer = decoder_gru2(layer,initial_state = initial_state)
    layer = decoder_gru3(layer,initial_state = initial_state)
    
    decoder_output = decoder_dense(layer)
    
    return decoder_output

In [0]:
decoder_output = connect_decoder(initial_state=encoder_output)

In [0]:
model_train = Model(inputs=[encoder_input,decoder_input],outputs=[decoder_output])

In [0]:
model_encoder = Model(inputs=[encoder_input],outputs=[encoder_output])

In [0]:
decoder_output = connect_decoder(initial_state = decoder_initial_state)

In [0]:
model_decoder = Model(inputs=[decoder_input,decoder_initial_state],outputs=[decoder_output])

In [0]:
def sparse_cross_entropy(y_true,y_pred):
    loss = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y_true,logits=y_pred)
    loss_mean = tf.reduce_mean(loss)
    return loss_mean

In [0]:
optimizer = RMSprop(lr=1e-3)

In [0]:
decoder_target = tf.placeholder(dtype='int32',shape=(None,None))

In [0]:
model_train.compile(optimizer=optimizer,
                   loss=sparse_cross_entropy,
                   target_tensors=[decoder_target])

In [0]:
path_checkpoint = 'checkpoint.keras'
checkpoint = ModelCheckpoint(filepath=path_checkpoint,save_weights_only=True)

In [144]:
# create on Colab directory
model_train.save('model_more.h5')    
model_file = drive.CreateFile({'title' : 'model_more.h5'})
model_file.SetContentFile('model_more.h5')
model_file.Upload()

# download to google drive
drive.CreateFile({'id': model_file.get('id')})

model_train.save_weights('model_weights_more.h5')
weights_file = drive.CreateFile({'title' : 'model_weights_more.h5'})
weights_file.SetContentFile('model_weights_more.h5')
weights_file.Upload()
drive.CreateFile({'id': weights_file.get('id')})

GoogleDriveFile({'id': '1QTYBhGcH-e_AO7WnX42n_4DOtWvun0aB'})

In [0]:



try:
  #  model_train.load_weights(path_checkpoint)
  # 3. reload weights from google drive into the model

  # use (get shareable link) to get file id
  last_weight_file = drive.CreateFile({'id': '1waobEqSnb2A5Cd0T_l5yC9Uo2Cwd6tNC'}) 
  last_weight_file.GetContentFile('last_weights_more.mat')
  model_train.load_weights('last_weights_more.mat')
except Exception as error:
    print('checkpoint yüklenemedi. eğitime baştan başlanıyor')
    print(error)
    

In [0]:
x_data = {'encoder_input': encoder_input_data,'decoder_input': decoder_input_data}


In [0]:
y_data = {'decoder_output': decoder_output_data}

In [157]:

model_train.fit(x=x_data,
               y=y_data,
               batch_size=64,
               epochs=10,
               callbacks=[checkpoint])
model_train.save('model_more.h5') 



Train on 568411 samples
Epoch 1/10


InvalidArgumentError: ignored

In [0]:
def CalculateRougeScores(refrence_summary,model_summary,scoring = False):
    rouge = RougeCalculator(stopwords=True, lang="en")
    rouge_1 = rouge.rouge_n(
            summary=model_summary,
            references=refrence_summary,
            n=1)
    rouge_2 = rouge.rouge_n(
            summary=model_summary,
            references=[refrence_summary],
            n=2)
    rouge_L = rouge.rouge_l(
            summary=model_summary,
            references=[refrence_summary])
    if (scoring == False):
        print("ROUGE-1: {}, ROUGE-2: {}, ROUGE-L: {}".format(
            rouge_1, rouge_2, rouge_L
        ).replace(", ", "\n"))
    
    return rouge_1,rouge_2,rouge_L

In [0]:
def CalculateBleuScores(refrence_summary,model_summary,scoring = False):
    refrence_summary_list = []
    rs =nltk.word_tokenize(refrence_summary)
    #print(r)
    refrence_summary_list.append(rs)
#     print(refrence_summary_list)
    model_summary = nltk.word_tokenize(model_summary)

#     print(model_summary)
#     a = [['good', 'quality', 'dog', 'food']]
#     b = ['a', 'good', 'dog', 'food']
    
    score = sentence_bleu(refrence_summary_list, model_summary,weights=(1, 0, 0, 0))
    if(scoring == False):
        print("BLEU Score: {} ".format(
            score,).replace(", ", "\n"))
        
    return score

In [0]:
def summarize(input_text, true_output_text=None,scoring = False):
    input_tokens = tokenizer_src.text_to_tokens(text=input_text,
                                                reverse=True,
                                                padding='pre')
    
    initial_state = model_encoder.predict(input_tokens)  
    max_tokens = tokenizer_dest.max_tokens    
    decoder_input_data = np.zeros(shape=(1, max_tokens), dtype=np.int)
    
        
    token_int = token_start
    output_text = ''
    count_tokens = 0
    
    while token_int != token_end and count_tokens < max_tokens:
        decoder_input_data[0, count_tokens] = token_int
        x_data = {'decoder_initial_state': initial_state, 'decoder_input': decoder_input_data}
        
        decoder_output = model_decoder.predict(x_data)
        
        token_onehot = decoder_output[0, count_tokens, :]
        token_int = np.argmax(token_onehot)
        
        sampled_word = tokenizer_dest.token_to_word(token_int)
        output_text += ' ' + sampled_word
        count_tokens += 1
        
    
       
    
    if scoring == False:
        print('Input text:')
        print(input_text)
        print()

        print('AI Summarized text:')
        print(output_text)
        print()
        if true_output_text is not None:
            print('Human Summarized text')
            print(true_output_text)
            print()
            
        rouge_1,rouge_2,rouge_L = CalculateRougeScores(true_output_text,output_text)
        score = CalculateBleuScores(true_output_text,output_text)
    else:
        rouge_1,rouge_2,rouge_L = CalculateRougeScores(true_output_text,output_text,True)
        score = CalculateBleuScores(true_output_text,output_text,True)  
    
    return rouge_1,rouge_2,rouge_L,score
    print()
    print()
    print()

In [0]:
summarize(input_text=data_src[73], true_output_text=data_dest[73])

In [0]:
def Scoring():    
    #pbar = ProgressBar()
    rouge_1_news = 0
    rouge_2_news = 0
    rouge_L_news = 0
    bleu_news = 0
    
    
    for i in range(1,20000):
        print(i)
        rouge_1,rouge_2,rouge_L,bleu = summarize(clean_texts[i],clean_summaries[i],True)
        rouge_1_news = rouge_1_news + rouge_1
        rouge_2_news = rouge_2_news + rouge_2
        rouge_L_news = rouge_L_news + rouge_L
        bleu_news = bleu_news + bleu              
    
    rouge_1_news_avg_score = rouge_1_news / i
    rouge_2_news_avg_score = rouge_2_news / i
    rouge_L_news_avg_score = rouge_L_news / i
    bleu_news_avg_score = bleu_news / i
    
    
    print("rouge_1_news: {}, rouge_2_news: {}, rouge_L_news: {}, bleu_news: {}".
          format(rouge_1_news_avg_score,rouge_2_news_avg_score,rouge_L_news_avg_score,bleu_news_avg_score).replace(", ", "\n"))
    
    
    
    

In [0]:
Scoring()